# Supplementary Results 14.1-14.2 — The genetic correlation matrix and therapeutic-area independence

What the matrix covers, and whether diseases within a therapeutic area are more genetically
correlated than diseases from two different areas — the assumption behind counting therapeutic areas
as a conservative measure of pleiotropy.

Numbers are written to `results/sr14_genetic_correlation.json`.

**Provenance.** `chapters/_legacy/06-review-r1/ta-independence/01_within_vs_between_ta.ipynb`: the
same single-area assignment by ontology descent, the same absolute correlations, and the same
permutation of area labels over diseases rather than over pairs. The LD score regression itself is
external; `01-data-preparation/13_genetic_correlation.ipynb` builds the matrix from its output.

**Not yet ported.** Supplementary Results 14.3 (effective independent traits) and 14.4 (disease-list
subsampling) remain in `chapters/_legacy/06-review-r1/effective-independent-traits/` and
`disease-subsampling/`. See the chapter README.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

from manuscript_methods import paper

numbers = {}
SEED = 20260813
PERMUTATIONS = 10000

matrix = pd.read_parquet(paper.derived("rg_matrix"))
print(f"matrix: {matrix.shape[0]} traits")

matrix: 1114 traits


## What the matrix holds and covers

In [2]:
# A trait counts as a measurement when the **upstream** `therapeutic_area` column of the canonical
# pairwise table says so. That is the labelling the published analysis used
# (`_legacy/06-review-r1/ta-independence/01_within_vs_between_ta.ipynb`, cell 3), and it reproduces
# 551 diseases and 563 measurements exactly. It is not the same as the trait's own ontology
# classification: 283 of the 1,114 traits carry no upstream label at all and count as diseases here,
# while `efo_therapeutic_area.primaryTherapeuticArea` puts them under the measurement root, giving
# 532/582. The matrix itself is built from this same file, so its labels travel with it.
pairwise = pd.read_parquet(
    paper.baseline("canonical_pairwise_table") + "/canonical_pairwise_table.parquet",
    columns=["diseaseId_1", "therapeutic_area_1", "diseaseId_2", "therapeutic_area_2"],
)
upstream = (
    pd.concat(
        [
            pairwise[["diseaseId_1", "therapeutic_area_1"]].set_axis(["trait", "area"], axis=1),
            pairwise[["diseaseId_2", "therapeutic_area_2"]].set_axis(["trait", "area"], axis=1),
        ]
    )
    .drop_duplicates("trait")
    .set_index("trait")["area"]
)
is_measurement = upstream.reindex(matrix.index).eq("measurement")

# The ontology-based alternative, for the record.
areas = pd.read_parquet(paper.derived("efo_therapeutic_area")).set_index("id")["primaryTherapeuticArea"]
by_term = areas.reindex(matrix.index) == paper.MEASUREMENT

numbers["S14.01"] = len(matrix)
numbers["S14.03"] = int(is_measurement.sum())
numbers["S14.02"] = len(matrix) - numbers["S14.03"]

off_diagonal = ~np.eye(len(matrix), dtype=bool)
measured = (matrix.to_numpy() != 0) & off_diagonal
numbers["S14.04"] = round(100 * measured.sum() / off_diagonal.sum(), 2)
print(f"traits {numbers['S14.01']} | diseases {numbers['S14.02']} | measurements {numbers['S14.03']}")
print(f"traits with no upstream label, counted as diseases: {int(upstream.reindex(matrix.index).isna().sum())}")
print(f"the term-level rule would give: diseases {int((~by_term).sum())} | measurements {int(by_term.sum())}")
print(f"off-diagonal entries with a measured correlation: {numbers['S14.04']}%")

traits 1114 | diseases 551 | measurements 563
traits with no upstream label, counted as diseases: 283
the term-level rule would give: diseases 532 | measurements 582
off-diagonal entries with a measured correlation: 99.84%


In [3]:
diseases = pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds"])
measurements = pd.read_parquet(paper.derived("prioritised_genes_measurements"), columns=["geneId", "diseaseIds"])
# The published association counts are over the genes of `gene_table`, the protein-coding set every
# gene-level analysis in this work uses: 115,017 gene-measurement associations, not the 150,360 the
# unrestricted prioritisation table holds. The disease side is unaffected, since all of its genes are
# already in that table. The *term* counts stay unrestricted, which is what the published table's own
# definition says ("unique disease or measurement ontology terms used in this work").
gene_table_genes = set(pd.read_parquet(paper.derived("gene_table"), columns=["geneId"])["geneId"])


def coverage(frame, label):
    """Terms and gene-trait associations, and how many of each the matrix covers."""
    exploded = frame.explode("diseaseIds").dropna(subset=["diseaseIds"]).drop_duplicates()
    terms = exploded["diseaseIds"].unique()
    exploded = exploded[exploded["geneId"].isin(gene_table_genes)]
    in_matrix = set(matrix.index)
    covered_terms = [t for t in terms if t in in_matrix]
    covered_rows = exploded[exploded["diseaseIds"].isin(in_matrix)]
    return [
        {"stratum": f"{label} terms", "total": len(terms), "in matrix": len(covered_terms)},
        {"stratum": f"gene-{label} associations", "total": len(exploded), "in matrix": len(covered_rows)},
    ]


coverage_table = pd.DataFrame(coverage(diseases, "disease") + coverage(measurements, "measurement"))
coverage_table["%"] = (100 * coverage_table["in matrix"] / coverage_table["total"]).round(1)

numbers["S14.05"] = int(coverage_table.loc[0, "in matrix"])
numbers["S14.06"] = float(coverage_table.loc[0, "%"])
numbers["S14.09"] = int(coverage_table.loc[1, "in matrix"])
numbers["S14.10"] = float(coverage_table.loc[1, "%"])
numbers["S14.07"] = int(coverage_table.loc[2, "in matrix"])
numbers["S14.08"] = float(coverage_table.loc[2, "%"])
numbers["S14.11"] = int(coverage_table.loc[3, "in matrix"])
numbers["S14.12"] = float(coverage_table.loc[3, "%"])
coverage_table

,stratum,total,in matrix,%
0,disease terms,1394,471,33.8
1,gene-disease associations,36858,27006,73.3
2,measurement terms,3412,507,14.9
3,gene-measurement associations,115017,86522,75.2


## Assigning each disease trait to a single therapeutic area

The first area whose ontology subtree contains the term, in the **legacy** ordering of the 23 roots —
the ordering the published analysis used for this section. Terms under no therapeutic-area root —
mostly phenotype codes sitting under a general phenotype term — are left out of the comparison.

In [4]:
ontology = pd.read_parquet(paper.release("disease") + "/disease.parquet", columns=["id", "descendants", "ancestors"])
roots = ontology[ontology["id"].isin(paper.THERAPEUTIC_AREAS)]
descendants = {row.id: set(list(row.descendants) if row.descendants is not None else []) for row in roots.itertuples()}
# The **legacy** ordering, which is the one the published analysis used here (it puts
# `genetic, familial or congenital disease` second to last rather than third). With the
# Supplementary Table 9 ordering the same 400 diseases spread over 22 areas instead of 21 and
# 55 pairs move from within-area to between-area. See `01-data-preparation/03_therapeutic_areas`.
priority = [key for key in paper.THERAPEUTIC_AREAS_LEGACY if key != paper.MEASUREMENT]


def single_area(term):
    """First therapeutic area in the published order whose subtree holds this term."""
    holding = {root for root, kids in descendants.items() if term == root or term in kids}
    for root in priority:
        if root in holding:
            return root
    return "other"


disease_traits = [t for t in matrix.index if not bool(is_measurement.get(t, False))]
area_of = {t: single_area(t) for t in disease_traits}
analysis_traits = [t for t in disease_traits if area_of[t] != "other"]

numbers["S14.13"] = len(analysis_traits)
numbers["S14.14"] = len(disease_traits) - len(analysis_traits)
numbers["S14.15"] = len({area_of[t] for t in analysis_traits})
print(
    f"disease traits in the matrix: {len(disease_traits)} | carrying an area: {numbers['S14.13']} | "
    f"under no area root: {numbers['S14.14']} | areas represented: {numbers['S14.15']}"
)

disease traits in the matrix: 551 | carrying an area: 400 | under no area root: 151 | areas represented: 21


## Within-area against between-area correlation

In [5]:
index = {t: i for i, t in enumerate(matrix.index)}
rows = np.array([index[t] for t in analysis_traits])
sub = matrix.to_numpy()[np.ix_(rows, rows)]
labels = np.array([area_of[t] for t in analysis_traits])
upper_i, upper_j = np.triu_indices(len(analysis_traits), 1)
correlation = sub[upper_i, upper_j]
same_area = labels[upper_i] == labels[upper_j]

# Pairs where one disease is a broader form of the other are near-identical by construction.
ancestors = {
    row.id: set(list(row.ancestors) if row.ancestors is not None else [])
    for row in ontology[ontology["id"].isin(analysis_traits)].itertuples()
}
terms = np.array(analysis_traits)
nested = np.array(
    [
        terms[j] in ancestors.get(terms[i], set()) or terms[i] in ancestors.get(terms[j], set())
        for i, j in zip(upper_i, upper_j)
    ]
)
print(
    f"disease pairs: {len(correlation):,} | within-area {int(same_area.sum()):,} | "
    f"between-area {int((~same_area).sum()):,} | nested {int(nested.sum()):,}"
)

disease pairs: 79,800 | within-area 5,844 | between-area 73,956 | nested 1,007


In [6]:
def compare(keep, label, draws=PERMUTATIONS, seed=SEED):
    """Within versus between comparison on one subset of pairs, with a label-permutation P value."""
    values = np.abs(correlation[keep])
    group = same_area[keep]
    within, between = values[group], values[~group]
    observed = float(within.mean() - between.mean())
    superiority = float(stats.mannwhitneyu(within, between, alternative="greater").statistic) / (
        len(within) * len(between)
    )
    fold = float((within >= 0.5).mean() / (between >= 0.5).mean())

    generator = np.random.default_rng(seed)
    extreme = 0
    for _ in range(draws):
        shuffled = generator.permutation(labels)
        permuted = (shuffled[upper_i] == shuffled[upper_j])[keep]
        if permuted.sum() == 0 or (~permuted).sum() == 0:
            continue
        if abs(values[permuted].mean() - values[~permuted].mean()) >= abs(observed):
            extreme += 1
    return {
        "pair set": label,
        "n_within": len(within),
        "n_between": len(between),
        "within": round(float(within.mean()), 3),
        "between": round(float(between.mean()), 3),
        "difference": round(observed, 3),
        ">=0.5 fold": round(fold, 2),
        "superiority": round(superiority, 3),
        "P": (extreme + 1) / (draws + 1),
    }


everything = np.ones(len(correlation), dtype=bool)
comparison = pd.DataFrame([compare(everything, "All disease pairs"), compare(~nested, "Excluding nested pairs")])
comparison

,pair set,n_within,n_between,within,between,difference,>=0.5 fold,superiority,P
0,All disease pairs,5844,73956,0.401,0.317,0.084,1.48,0.574,0.0001
1,Excluding nested pairs,5103,73690,0.390,0.316,0.073,1.42,0.563,0.0001


In [7]:
numbers["S14.16"] = int(comparison.loc[0, "n_within"])
numbers["S14.17"] = int(comparison.loc[0, "n_between"])
numbers["S14.18"] = float(comparison.loc[0, "within"])
numbers["S14.19"] = float(comparison.loc[0, "between"])
numbers["S14.20"] = float(comparison.loc[0, "difference"])
numbers["S14.21"] = float(comparison.loc[0, ">=0.5 fold"])
numbers["S14.22"] = float(comparison.loc[0, "superiority"])
numbers["S14.23"] = int(nested.sum())
numbers["S14.24"] = float(comparison.loc[1, "within"])
numbers["S14.25"] = float(comparison.loc[1, "between"])
numbers["S14.26"] = float(comparison.loc[1, ">=0.5 fold"])
numbers["S14.27"] = float(comparison.loc[1, "superiority"])
numbers["S14.28"] = int(comparison.loc[1, "n_within"])
numbers["S14.29"] = int(comparison.loc[1, "n_between"])
numbers["S14.30"] = round(100 * float((np.abs(correlation[~same_area]) >= 0.5).mean()), 0)
print(f"between-area pairs above 0.5: {numbers['S14.30']}%")
print({k: numbers[k] for k in ["S14.18", "S14.19", "S14.20", "S14.21", "S14.22"]})

between-area pairs above 0.5: 22.0%
{'S14.18': 0.401, 'S14.19': 0.317, 'S14.20': 0.084, 'S14.21': 1.48, 'S14.22': 0.574}


## Write the results

In [8]:
print(paper.save_results("sr14_genetic_correlation", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/sr14_genetic_correlation.json


,computed
S14.01,1114.000
S14.03,563.000
S14.02,551.000
S14.04,99.840
S14.05,471.000
S14.06,33.800
S14.09,27006.000
S14.10,73.300
S14.07,507.000
S14.08,14.900
